# SEBAL Notebook 4: Sensible Heat Flux (H) and dT Calibration  Workflow
# SEBAL Sensible Heat Flux Estimation Workflow
**Scene:** Landsat 7 (2020-01-19)  
**Region:** Mississippi Delta (EPSG:32615)

This notebook continues from **04_sebal_surface_indices**.  
Here we compute aerodynamic resistance (rah), calibrate near-surface temperature difference (dT) using hot and cold anchor pixels, and estimate sensible heat flux (H) for the SEBAL workflow.

## 1. Import Required Python Libraries and Utility Functions

In [ ]:
import os
import rasterio
import numpy as np
from Utility import get_file_dataframe
from Utility import load_rasters
from Utility import calculate_NDVI
from Utility import calculate_albedo
from Utility import extract_mask_mean
from Utility import read_raster
from Utility import write_raster
from Utility import check_raster

## 2. Set SEBAL working directory

In [ ]:
os.chdir("..")
os.getcwd()

## 3. Define bands, directories and input files

In [ ]:
clip_dir = "03_clip_landsat"
# output folder
out_dir = "04_indices"
os.makedirs(out_dir, exist_ok=True)
region = 'MSDelta'
proc_dir = "03_processed_met"
hour_str = '16'
bands = ["blue","green","red","nir08","swir16","swir22","lwir"]
landsat_file = "landsat_downloaded_files.csv"

## 4. Load clipped rasters

In [ ]:
rasters, srcs, date = load_rasters(landsat_file, clip_dir, bands)

## 5. Normalized Vedigtation Difference Index (NDVI)
### 5.1 Calculate NDVI
$$
NDVI = \frac{NIR - RED}{NIR + RED}
$$

In [ ]:
ndvi = calculate_NDVI(rasters)

### 5.2 Save NDVI

In [ ]:
# output filename
out_path = f"{out_dir}/NDVI_{date}_{region}.tif"
# use metadata from any band source (red is fine)
meta = srcs["red"].meta.copy()
write_raster(raster_path=out_path, profile=meta, value=ndvi)

### 5.3 Quick validation

In [ ]:
ndvi_check = rasterio.open(out_path).read(1)
print("shape:", ndvi_check.shape)
print("min:", ndvi_check[ndvi_check > -9999].min())
print("max:", ndvi_check[ndvi_check > -9999].max())

## 6. Surface Albedo (α)
### 6.1 Calculate Albedo
$$
\alpha = 0.356 \rho_{BLUE} + 0.130 \rho_{RED}  + 0.373 \rho_ {NIR} + 0.085 \rho_{SWIR1}  + 0.072 \rho_{SWIR2} - 0.0018
$$
where SWIR1 = Shortwave Infrared Band 1, SWIR2 = Shortwave Infrared Band 2

In [ ]:
albedo = calculate_albedo(rasters)

### 6.2 Save Albedo

In [ ]:
# output filename
out_path = f"{out_dir}/Albedo_raw_{date}_{region}.tif"
write_raster(raster_path=out_path, profile=meta, value=albedo)

### 6.3 Quick validation

In [ ]:
albedo_check = rasterio.open(out_path).read(1)
print("shape:", albedo_check.shape)
print("min:", albedo_check[albedo_check > -9999].min())
print("max:", albedo_check[albedo_check > -9999].max())

### 6.4 Final SEBAL Albedo

In [ ]:
# SR scaling for Landsat Collection 2 Level-2
scale  = 0.0000275
offset = -0.2

blue_ref  = rasters["blue"].astype("float32")  * scale + offset
red_ref   = rasters["red"].astype("float32")   * scale + offset
nir_ref   = rasters["nir08"].astype("float32") * scale + offset
swir1_ref = rasters["swir16"].astype("float32")* scale + offset
swir2_ref = rasters["swir22"].astype("float32")* scale + offset

valid = (blue_ref > 0) & (red_ref > 0) & (nir_ref > 0) & (swir1_ref > 0) & (swir2_ref > 0)

albedo_ref = np.full(red_ref.shape, -9999.0, dtype=np.float32)
albedo_ref[valid] = (
    0.356*blue_ref[valid] +
    0.130*red_ref[valid] +
    0.373*nir_ref[valid] +
    0.085*swir1_ref[valid] +
    0.072*swir2_ref[valid] -
    0.0018
)

print("Albedo (reflectance) min/max:", albedo_ref[valid].min(), albedo_ref[valid].max())

In [ ]:
out_path_final = f"{out_dir}/ALBEDO_{date}_{region}.tif"
write_raster(raster_path=out_path_final, profile=meta, value=albedo_ref)

In [ ]:
albedo_ref[valid] = np.clip(albedo_ref[valid], 0.02, 0.35)

print("Clamped min/max:", 
      albedo_ref[valid].min(), 
      albedo_ref[valid].max())

## 7. LST (Land Surface Temperature)
### 7.1 Locate the LST band

In [ ]:
# Locate LWIR raster
lwir_tif = os.path.join(clip_dir, f"lwir_{date}_{region}.tif")

# read raster
lst_raw, profile, nod = read_raster(lwir_tif)

# assign fallback nodata
nod = nod if nod is not None else -9999.0

### 7.2 Convert to Celsius
The Landsat Level-2 surface temperature is stored as scaled digital numbers (DN).  
Conversion follows:
$$
T_K = DN \times 0.00341802 + 149.0
$$
$$
T_C = T_K - 273.15
$$

In [ ]:
# Convert Landsat Collection-2 LST stored integers to Celsius

scale = 0.00341802
offset = 149.0

valid = (lst_raw != nod)

lst_k = np.full(lst_raw.shape, -9999.0, dtype="float32")
lst_k[valid] = lst_raw[valid] * scale + offset

lst_c = np.full(lst_raw.shape, -9999.0, dtype="float32")
lst_c[valid] = lst_k[valid] - 273.15

print("LST C min/max:",
      lst_c[lst_c > -9999].min(),
      lst_c[lst_c > -9999].max())

### 7.3 Save the  LST

In [ ]:
# output filename
out_path = f"{out_dir}/LST_{date}_{region}.tif"
write_raster(raster_path=out_path, profile=meta, value=lst_c)

### 7.4 Mask LST using NDVI > 0

In [ ]:
ndvi_path = f"{out_dir}/NDVI_{date}_{region}.tif"
ndvi, ndvi_profile, ndvi_nod = read_raster(ndvi_path)

mask = (ndvi > 0.0) & (lst_c > -9999)
lst_c2 = np.full(lst_c.shape, -9999.0, dtype="float32")
lst_c2[mask] = lst_c[mask]

print("Masked LST C min/max:", 
      lst_c2[lst_c2 > -9999].min(), 
      lst_c2[lst_c2 > -9999].max())

### 7.5 Save the masked LST

In [ ]:
out_path2 = f"{out_dir}/LST_C_{date}_{region}.tif"
write_raster(raster_path=out_path2, profile=meta, value=lst_c2)

### 7.6 Check

In [ ]:
print("Valid LST before mask:", np.sum(lst_c > -9999))
print("Valid LST after mask: ", np.sum(lst_c2 > -9999))
print("Removed pixels:       ", np.sum((lst_c > -9999) & (lst_c2 == -9999)))

## 8. Net Radiation (SEBAL)

Net radiation is computed as:

$$
R_n = R_s^\downarrow (1 - \alpha) + R_L^\downarrow - R_L^\uparrow
$$

where:

• $R_s^\downarrow$ = incoming shortwave radiation (AORC)  
• $\alpha$ = surface albedo  
• $R_L^\downarrow$ = incoming longwave radiation  
• $R_L^\uparrow$ = outgoing longwave radiation
### 8.1 Load inputs

In [ ]:
# ---- file paths
rs_path   = f"{proc_dir}/Rs_{date}_{hour_str}Z_{region}.tif"
ta_path   = f"{proc_dir}/TA_{date}_{hour_str}Z_{region}.tif"
alb_path  = f"{out_dir}/ALBEDO_{date}_{region}.tif"
lst_path  = f"{out_dir}/LST_C_{date}_{region}.tif"    # <- masked LST preferred
ndvi_path = f"{out_dir}/NDVI_{date}_{region}.tif"

# ---- read rasters
# read Rs raster
Rs, rs_profile, rs_nodata = read_raster(rs_path)

# read TA raster
TA, ta_profile, ta_nodata = read_raster(ta_path)

# read albedo raster
alb, alb_profile, alb_nodata = read_raster(alb_path)

# read LST raster
lstC, lst_profile, lst_nodata = read_raster(lst_path)

# read NDVI raster
ndvi, ndvi_profile, ndvi_nodata = read_raster(ndvi_path)

# ---- valid mask
valid = (Rs > -9999) & (TA > -9999) & (alb > -9999) & (lstC > -9999)

print("Valid pixels:", int(valid.sum()))

### 8.2 Compute Rn
- Compute net radiation (Rn) using a simplified SEBAL formulation.
- Surface albedo is taken from broadband albedo, surface temperature is taken from LST,
- and surface emissivity is assigned from NDVI-based classes.
- Incoming longwave radiation is approximated using the Stefan-Boltzmann law with air temperature,
- while outgoing longwave radiation is computed from surface emissivity and surface temperature.
- This is a simplified implementation intended for stable first-pass SEBAL calculations.

In [ ]:
sigma = 5.67e-8

# create empty Kelvin air-temperature array (same size as TA)
TaK  = np.full(TA.shape, np.nan, dtype="float32")
 # create empty Kelvin surface-temperature array (same size as LST)
LstK = np.full(lstC.shape, np.nan, dtype="float32")

TaK[valid]  = TA[valid]              # copy valid TA pixels directly (already in Kelvin)
LstK[valid] = lstC[valid] + 273.15   # convert valid LST pixels from Celsius → Kelvin

RL_down = np.full(Rs.shape, np.nan, dtype="float32") # allocate array for incoming longwave radiation
RL_up   = np.full(Rs.shape, np.nan, dtype="float32") # allocate array for outgoing longwave radiation

# --- emissivity (NDVI-based, stable SEBAL version)
emiss = np.full(ndvi.shape, np.nan, dtype="float32")

# SEBAL simple scheme
# assign high emissivity for dense vegetation pixels
emiss[(ndvi > 0.5)] = 0.99
# assign moderate emissivity for mixed vegetation pixels
emiss[(ndvi > 0.2) & (ndvi <= 0.5)] = 0.97
# assign lower emissivity for bare soil / sparse vegetation
emiss[(ndvi <= 0.2)] = 0.95

# compute incoming longwave radiation using Stefan–Boltzmann law
RL_down[valid] = sigma * (TaK[valid] ** 4)
# compute outgoing longwave using emissivity + surface temperature
RL_up[valid]   = emiss[valid] * sigma * (LstK[valid] ** 4)


 # initialize net radiation array with nodata
Rn = np.full(Rs.shape, -9999.0, dtype="float32")
 # compute net radiation (SEBAL equation)
Rn[valid] = Rs[valid] * (1.0 - alb[valid]) + RL_down[valid] - RL_up[valid]

# quick range check for sanity
print("Rn valid:", np.sum(Rn > -9999))
print("FIXED Rn min/max:", float(Rn[Rn>-9999].min()), float(Rn[Rn>-9999].max()))

### 8.3 Save Rn

In [ ]:

# define output filename for net radiation raster
out_rn = f"{out_dir}/Rn_{date}_{hour_str}Z_{region}.tif"
write_raster(raster_path=out_rn, profile=rs_profile, value=Rn)

##  9 Soil Heat Flux (G)

SEBAL uses an empirical relationship. The most common form (Bastiaanssen-type) is:

$$
𝐺 = 𝑅_𝑛 \frac{𝑇_𝑠}{\alpha} (0.0038𝛼 + 0.0074𝛼^2) (1 − 0.98𝑁𝐷𝑉𝐼^4)
$$

### 9.1 Load inputs (Rn, LST, Albedo, NDVI)

In [ ]:
# define path to net radiation raster
rn_path   = f"{out_dir}/Rn_{date}_{hour_str}Z_{region}.tif"

# define path to surface temperature raster (Celsius)
lst_path  = f"{out_dir}/LST_C_{date}_{region}.tif"

# define path to surface albedo raster
alb_path  = f"{out_dir}/ALBEDO_{date}_{region}.tif"

# define path to NDVI raster
ndvi_path = f"{out_dir}/NDVI_{date}_{region}.tif"


## load net radiation raster
Rn, profile, rn_nodata = read_raster(rn_path)

# load surface temperature raster (°C)
LstC, _, lst_nodata = read_raster(lst_path)

# load albedo raster
Alb, _, alb_nodata = read_raster(alb_path)

# load NDVI raster
Ndvi, _, ndvi_nodata = read_raster(ndvi_path)

# create physically valid NDVI copy for anchor-pixel selection
Ndvi2 = np.where((Ndvi > -1.0) & (Ndvi < 1.0), Ndvi, np.nan)

# build mask of valid pixels across all inputs
valid = (Rn > -9999) & (LstC > -9999) & (Alb > -9999) & (Ndvi > -9999)

# print count for quick verification
print("Valid pixels:", int(valid.sum()))

### 9.2 Compute G

In [ ]:
# initialize G raster
G = np.full(Rn.shape, -9999.0, dtype="float32")

# protect against very small albedo values
alb_safe = np.where((alb > 0.01) & valid, alb, np.nan)

# use cleaned NDVI
ndvi_safe = np.where(valid, Ndvi2, np.nan)

# compute SEBAL soil heat flux using LST in Celsius
G_calc = Rn * (
    LstC *
    (0.0038 + 0.0074 * alb_safe) *
    (1 - 0.98 * ndvi_safe**4)
)

# assign valid pixels
G[valid] = G_calc[valid]

# clean valid mask
g_valid = valid & np.isfinite(G) & (G > -9999) & np.isfinite(Rn) & (Rn > 0)

# diagnostics
ratio = np.full(Rn.shape, np.nan, dtype="float32")
ratio[g_valid] = G[g_valid] / Rn[g_valid]

### 9.3 Sanity check

In [ ]:
# print clean diagnostics
print("Raw G valid:", np.sum(g_valid))
print("Raw G min/max:", float(G[g_valid].min()), float(G[g_valid].max()))
print("Raw G/Rn min/max:", float(np.nanmin(ratio)), float(np.nanmax(ratio)))

### 9.4 Clamp G (recommended)

In [ ]:
# Apply a practical physical constraint to G/Rn:
# enforce nonnegative soil heat flux and cap unrealistically high ratios
# before saving the final soil heat flux raster.
ratio_clamped = np.clip(ratio, 0.0, 0.35)

# build final soil heat flux raster
G2 = np.full(Rn.shape, -9999.0, dtype="float32")
G2[g_valid] = Rn[g_valid] * ratio_clamped[g_valid]

# final diagnostics
print("G2 valid:", np.sum(G2 > -9999))
print("G2 min/max:", float(G2[G2 > -9999].min()), float(G2[G2 > -9999].max()))
print("G2/Rn min/max:",
      float(np.nanmin(ratio_clamped)),
      float(np.nanmax(ratio_clamped)))

### 9.5 Save G

In [ ]:
# define output filename for soil heat flux raster
out_g = f"{out_dir}/G_{date}_{hour_str}Z_{region}.tif"
G = G2
write_raster(raster_path=out_g, profile=profile, value=G)

## 10. Sensible Heat Flux (H)
### 10.1 Load Inputs

In [ ]:
# define paths for required rasters
rn_path   = f"{out_dir}/Rn_{date}_{hour_str}Z_{region}.tif"
g_path    = f"{out_dir}/G_{date}_{hour_str}Z_{region}.tif"
lst_path  = f"{out_dir}/LST_C_{date}_{region}.tif"
ndvi_path = f"{out_dir}/NDVI_{date}_{region}.tif"

# load net radiation raster
Rn, profile, rn_nodata = read_raster(rn_path)

# load soil heat flux raster
G, _, g_nodata = read_raster(g_path)

# load land surface temperature raster
LstC, _, lst_nodata = read_raster(lst_path)

# load NDVI raster
Ndvi, _, ndvi_nodata = read_raster(ndvi_path)

# build valid mask
valid = (Rn > -9999) & (G > -9999) & (LstC > -9999) & (Ndvi > -9999)

print("Valid pixels:", int(valid.sum()))

### 10.2 Prepare clean mask for anchor selection

In [ ]:
# refined valid mask for anchor selection
anchor_valid = (
    (valid) &
    np.isfinite(Rn) & np.isfinite(G) &
    np.isfinite(LstC) & np.isfinite(Ndvi)
)

print("Anchor-valid pixels:", int(anchor_valid.sum()))

In SEBAL, the hot and cold pixels are selected to represent the two physical extremes of surface energy balance needed to calibrate sensible heat flux (H). The cold pixel represents well-watered vegetation with strong evapotranspiration, so it must have relatively high NDVI and low land surface temperature (LST). The hot pixel represents dry bare soil or sparse vegetation with minimal evapotranspiration, so it must have low NDVI and high LST, following the standard SEBAL framework (Bastiaanssen et al., 1998). Thresholds such as NDVI < 0.2 or NDVI > 0.35 (for winter scenes) reflect realistic vegetation conditions in the scene rather than fixed rules.

In practice, anchor selection follows a polarized logic for stability. Cold pixels (preferred) include dense or moist vegetation with relatively cool LST and uniform surfaces, while hot pixels (preferred) include dry bare soil with warm LST. Percentile-based filtering is commonly used to avoid outliers and improve robustness in operational applications (Allen et al., 2007).

*References*

Allen, R.G., Tasumi, M., & Trezza, R. (2007). Satellite-based energy balance for mapping evapotranspiration with internalized calibration (METRIC). Journal of Irrigation and Drainage Engineering, 133(4), 380–394.

Bastiaanssen, W.G.M., Menenti, M., Feddes, R.A., & Holtslag, A.A.M. (1998). A remote sensing surface energy balance algorithm for land (SEBAL). Journal of Hydrology, 212–213, 198–212.
### 10.3 Prepare for Cold Pixel Selection

In [ ]:
# Restrict cold-pixel search to vegetated surfaces
# and retain the coolest 10% of LST within that vegetation mask.# use cleaned NDVI for anchor selection
ndvi_anchor = Ndvi2

# restrict to dense vegetation
cold_mask = anchor_valid & (ndvi_anchor >= 0.35)

print("Cold vegetation pixels:", int(cold_mask.sum()))

# pick lowest temperature inside vegetation
cold_LST_thr = np.nanpercentile(LstC[cold_mask], 10)

cold_pixels = cold_mask & (LstC <= cold_LST_thr)
print("Cold candidates:", int(cold_pixels.sum()))
print("Cold NDVI:", float(np.nanmean(Ndvi[cold_pixels])))
print("Cold LST:", float(np.nanmean(LstC[cold_pixels])))
print("Cold albedo:", float(np.nanmean(Alb[cold_pixels])))
print("Cold G/Rn:", float(np.nanmean((G / Rn)[cold_pixels])))

### 10.4 Refine cold candidates

In [ ]:
cold_pixels2 = cold_pixels & (Ndvi >= 0.45)

print("Refined cold count:", int(cold_pixels2.sum()))
print("Refined cold NDVI:", float(np.nanmean(Ndvi[cold_pixels2])))
print("Refined cold LST:", float(np.nanmean(LstC[cold_pixels2])))
print("Refined cold albedo:", float(np.nanmean(Alb[cold_pixels2])))
print("Refined cold G/Rn:", float(np.nanmean((G / Rn)[cold_pixels2])))

### 10.5 Final cold tightening (temperature only)

In [ ]:
# Final cold-anchor pool:
# dense vegetation with low surface temperature, low albedo,
# and very low G/Rn, consistent with a well-watered cold pixel.
cold_LST_thr2 = np.nanpercentile(LstC[cold_pixels2], 5)
cold_pixels_final = cold_pixels2 & (LstC <= cold_LST_thr2)

print("Final cold count:", int(cold_pixels_final.sum()))
print("Final cold NDVI:", float(np.nanmean(Ndvi[cold_pixels_final])))
print("Final cold LST:", float(np.nanmean(LstC[cold_pixels_final])))
print("Final cold albedo:", float(np.nanmean(Alb[cold_pixels_final])))
print("Final cold G/Rn:", float(np.nanmean((G / Rn)[cold_pixels_final])))

### 10.6 Select Hot Pixel

In [ ]:
# use cleaned NDVI for anchor selection
ndvi_anchor = Ndvi2

# restrict to sparse vegetation / bare soil
hot_mask = anchor_valid & (ndvi_anchor <= 0.2)

print("Hot dry pixels:", int(hot_mask.sum()))

# pick hottest part of dry pixels
hot_LST_thr = np.nanpercentile(LstC[hot_mask], 90)
# Final hot-anchor pool:
# sparse vegetation/bare soil with highest LST,
# low NDVI, moderate G/Rn, consistent with dry hot surfaces.
hot_pixels = hot_mask & (LstC >= hot_LST_thr)
hot_pixels2 = hot_pixels & (Ndvi <= 0.15)
hot_LST_thr2 = np.nanpercentile(LstC[hot_pixels2], 95)
hot_pixels_final = hot_pixels2 & (LstC >= hot_LST_thr2)
print("Hot candidates:", int(hot_pixels.sum()))
print("Hot NDVI:", float(np.nanmean(ndvi_anchor[hot_pixels])))
print("Hot LST:", float(np.nanmean(LstC[hot_pixels])))
print("Hot albedo:", float(np.nanmean(Alb[hot_pixels])))
print("Hot G/Rn:", float(np.nanmean((G / Rn)[hot_pixels])))
print("Final hot count:", int(hot_pixels_final.sum()))
print("Final hot NDVI:", float(np.nanmean(Ndvi[hot_pixels_final])))
print("Final hot LST:", float(np.nanmean(LstC[hot_pixels_final])))
print("Final hot albedo:", float(np.nanmean(Alb[hot_pixels_final])))
print("Final hot G/Rn:", float(np.nanmean((G / Rn)[hot_pixels_final])))

### 10.7 Save hot/cold masks

In [ ]:
# convert masks to raster form
hot_mask_raster = np.full(Rn.shape, 0, dtype="int16")
cold_mask_raster = np.full(Rn.shape, 0, dtype="int16")

hot_mask_raster[hot_pixels_final] = 1
cold_mask_raster[cold_pixels_final] = 1

# save hot mask
hot_out = f"{out_dir}/hot_pixels_{date}_{hour_str}Z_{region}.tif"
write_raster(raster_path=hot_out, profile=meta, value=hot_mask_raster)

# save cold mask
cold_out = f"{out_dir}/cold_pixels_{date}_{hour_str}Z_{region}.tif"
write_raster(raster_path=cold_out, profile=meta, value=cold_mask_raster)

## 11. dT Calibration
### 11.1 Extract masks

In [ ]:
# example (edit names as needed)
hot_mask  = os.path.join(out_dir, f"hot_pixels_{date}_{hour_str}Z_{region}.tif")
cold_mask  = os.path.join(out_dir, f"cold_pixels_{date}_{hour_str}Z_{region}.tif")

rasters = {
    "LST":  f"{out_dir}/LST_{date}_{region}.tif",
    "Rn": f"{out_dir}/Rn_{date}_{hour_str}Z_{region}.tif",
    "G": f"{out_dir}/G_{date}_{hour_str}Z_{region}.tif",
    "NDVI": f"{out_dir}/NDVI_{date}_{region}.tif",
    "Albedo": f"{out_dir}/ALBEDO_{date}_{region}.tif"
}

# Dictionary to store anchor values
anchors = {}

for name, path in rasters.items():
    hot_val  = extract_mask_mean(path, hot_mask)    # mean over hot mask
    cold_val = extract_mask_mean(path, cold_mask)   # mean over cold mask

    # store values for later steps
    anchors[name] = {"hot": hot_val, "cold": cold_val}

    # keep print for verification
    print(name, "| HOT:", hot_val, "| COLD:", cold_val)

### 11.2 Compute H at hot and cold pixels
We now compute sensible heat flux H at anchors:
$$
𝐻 = 𝑅_𝑛 − 𝐺 − 𝐿𝐸
$$

In SEBAL anchor logic:

Cold pixel, assume H ≈ 0

Hot pixel, assume LE ≈ 0, so:
$$
𝐻_{ℎot} = 𝑅_{𝑛,ℎ𝑜𝑡} − 𝐺_{ho𝑡}
$$

In [ ]:
# 11.2 Convert LST to Kelvin for calibration
Ts_hot  = anchors["LST"]["hot"] + 273.15
Ts_cold = anchors["LST"]["cold"] + 273.15

# Net radiation at hot pixel (W/m²) from anchor extraction
Rn_hot = anchors["Rn"]["hot"] 

# Soil heat flux at hot pixel (W/m²)
G_hot  = anchors["G"]["hot"]

# Sensible heat flux at hot pixel (assume LE ≈ 0 for hot pixel)
H_hot = Rn_hot - G_hot  

# Sensible heat flux at cold pixel (assume H ≈ 0 for cold pixel)
H_cold = 0.0  

# Print results for verification before moving to dT calibration
print("Ts_hot:", Ts_hot)
print("Ts_cold:", Ts_cold)
print("H_hot:", H_hot)

### 11.3 Solve linear dT relation

In SEBAL
$$
dT = a T_b + b
$$
where, 
$$
a = \frac{dT_{hot} - dT_{cold}}{T_{s,hot} - T_{s,cold}}
$$
$$
b = dT_{hot} - a\,T_{s,hot}
$$

For SEBAL anchor calibration:

- Cold pixel: $dT_{cold} = 0$
- Hot pixel: $dT_{hot}$ will be computed from the anchor energy balance

In [ ]:
rah_hot = 110.0   # s/m, initial neutral assumption
rho_cp = 1.25 * 1004   # J/m3/K

dT_hot = H_hot * rah_hot / rho_cp
dT_cold = 0.0

a = (dT_hot - dT_cold) / (Ts_hot - Ts_cold)
b = dT_hot - a * Ts_hot

print("dT_hot:", dT_hot)
print("a:", a)
print("b:", b)

### 11.3B Load corrected rah and recompute dT relation

In [ ]:
# Run after Notebook 5
rahcorr_path = os.path.join("04_indices", f"rah_corr_{date}_{hour_str}Z_{region}.tif")

rah_corr_fixed, _, _ = read_raster(rahcorr_path)

print("rah_corr_fixed min/max:",
      float(rah_corr_fixed[rah_corr_fixed > -9999].min()),
      float(rah_corr_fixed[rah_corr_fixed > -9999].max()))

# corrected rah at hot anchor
rah_hot_corr = float(np.nanmean(rah_corr_fixed[hot_pixels]))

rho_cp = 1.25 * 1004   # J/m3/K

dT_hot_corr = H_hot * rah_hot_corr / rho_cp
dT_cold_corr = 0.0

a_corr = (dT_hot_corr - dT_cold_corr) / (Ts_hot - Ts_cold)
b_corr = dT_hot_corr - a_corr * Ts_hot

print("rah_hot_corr:", rah_hot_corr)
print("dT_hot_corr:", dT_hot_corr)
print("a_corr:", a_corr)
print("b_corr:", b_corr)

### 11.4 Compute calibrated dT raster
$$
dT_{raster} = aT_s + b
$$

In [ ]:
# # Compute first-pass dT from the anchor-based linear relation
# and constrain to a nonnegative physical range before saving.
dT = np.full(LstC.shape, -9999.0, dtype="float32")

valid_dt = (LstC > -9999)
Ts_full = LstC + 273.15

dT[valid_dt] = a * Ts_full[valid_dt] + b

print("dT min/max:", float(dT[dT > -9999].min()), float(dT[dT > -9999].max()))

# Clamp first-pass dT to physical lower bound

dT2 = np.full(LstC.shape, -9999.0, dtype="float32")
dT2[valid_dt] = np.maximum(dT[valid_dt], 0.0)

print("Clamped dT min/max:",
      float(dT2[dT2 > -9999].min()),
      float(dT2[dT2 > -9999].max()))


### 11.4B Compute calibrated dT using corrected coefficients

In [ ]:
# Run after Notebook 5
dT_corr = np.full(LstC.shape, -9999.0, dtype="float32")

valid_dt = (LstC > -9999)

Ts_full = LstC + 273.15

dT_corr[valid_dt] = a_corr * Ts_full[valid_dt] + b_corr

print("dT_corr min/max:",
      float(dT_corr[dT_corr > -9999].min()),
      float(dT_corr[dT_corr > -9999].max()))

# clamp to physical lower bound
dT_corr2 = np.full(LstC.shape, -9999.0, dtype="float32")
dT_corr2[valid_dt] = np.maximum(dT_corr[valid_dt], 0.0)

print("Clamped dT_corr min/max:",
      float(dT_corr2[dT_corr2 > -9999].min()),
      float(dT_corr2[dT_corr2 > -9999].max()))

### 10.5 Save calibrated dT raster

In [ ]:
profile.update(dtype="float32", nodata=-9999.0)

dT_out = np.where(dT2 > -9999, dT2, -9999).astype("float32")

dT_path = f"{out_dir}/dT_{date}_{hour_str}Z_{region}.tif"

write_raster(raster_path=dT_path, profile=profile, value=dT_out)


### 11.5B Save corrected dT raster

In [ ]:
profile.update(dtype="float32", nodata=-9999.0)

dTcorr_out = np.where(dT_corr2 > -9999, dT_corr2, -9999).astype("float32")

dTcorr_path = f"04_indices/dT_corr_{date}_{hour_str}Z_{region}.tif"

write_raster(raster_path=dTcorr_path, profile=profile, value=dTcorr_out)


### 11.6 Compute first-pass H raster
The sensible heat flux relation is:

$$
H = \rho \, C_p \, \frac{dT}{r_{ah}}
$$

So the aerodynamic resistance is:

$$
r_{ah} = \frac{\rho \, C_p \, dT}{H}
$$

In [ ]:
H1 = np.full(dT2.shape, -9999.0, dtype="float32")

valid_H = (dT2 > -9999)
H1[valid_H] = rho_cp * dT2[valid_H] / rah_hot

print("H1 min/max:",
      float(H1[H1 > -9999].min()),
      float(H1[H1 > -9999].max()))

# Save H
H_out = os.path.join(out_dir, f"H_{date}_{region}.tif")

write_raster(H_out, profile, H1)

### 11.7 Compute first-pass LE raster

In [ ]:
LE1 = np.full(H1.shape, -9999.0, dtype="float32")

valid_LE = (H1 > -9999)
LE1[valid_LE] = Rn[valid_LE] - G[valid_LE] - H1[valid_LE]

print("LE1 min/max:",
      float(LE1[LE1 > -9999].min()),
      float(LE1[LE1 > -9999].max()))

# Clamp first-pass LE to a physical range

LE2 = np.full(LE1.shape, -9999.0, dtype="float32")
valid_LE2 = (LE1 > -9999)
LE2[valid_LE2] = np.maximum(LE1[valid_LE2], 0.0)

print("Clamped LE min/max:",
      float(LE2[LE2 > -9999].min()),
      float(LE2[LE2 > -9999].max()))


### 11.8 Compute first-pass evaporative fraction

In [ ]:
EF1 = np.full(LE2.shape, -9999.0, dtype="float32")

denom = Rn - G
valid_EF = (LE2 > -9999) & (denom > 0)

EF1[valid_EF] = LE2[valid_EF] / denom[valid_EF]

print("EF1 min/max:",
      float(EF1[EF1 > -9999].min()),
      float(EF1[EF1 > -9999].max()))

### 11.9 Compute instantaneous ET equivalent (mm/hr)

------------------------------------------------------------
**FIRST-PASS INSTANTANEOUS ET (SEBAL)**

This section computes first-pass instantaneous evapotranspiration (ETinst) using the SEBAL energy balance framework.

*Steps followed:*
1. Sensible heat flux (H) estimated using anchor-calibrated dT and first-pass aerodynamic resistance (neutral assumption).
2. Latent heat flux (LE) computed as: $LE = Rn − G − H$
3. Evaporative fraction (EF) derived: $EF = LE / (Rn − G)$
4. Instantaneous ET estimated via: $ETinst = EF × ETeq$   where: $ETeq = (Rn − G) × 3600 / λ$
*Note:*
This is a FIRST-PASS ET solution. At this stage:
- aerodynamic resistance is not stability-corrected,
- LE may contain numerical spikes from anchor assumptions,
- meteorological refinement is not yet applied.

Therefore, ETinst is constrained to a physically realistic range (0–1.5 mm/hr) as quality control. This clipping is not empirical modeling but stabilization of the first-pass solution.

Final ET will later be refined through stability-corrected aerodynamic resistance and iterative SEBAL adjustment.

------------------------------------------------------------

In [ ]:
# Final instantaneous ET (mm/hr)

lambda_v = 2.45e6

ETeq = np.full(Rn.shape, -9999.0, dtype="float32")
valid_eq = (Rn - G) > 0
ETeq[valid_eq] = (Rn[valid_eq] - G[valid_eq]) * 3600 / lambda_v

ETinst = np.full(EF1.shape, -9999.0, dtype="float32")
valid_inst = (EF1 > -9999) & (ETeq > -9999)

ETinst[valid_inst] = EF1[valid_inst] * ETeq[valid_inst]

# Clamp to realistic range
ETinst[valid_inst] = np.clip(ETinst[valid_inst], 0.0, 1.5)

print("Final ETinst min/max:",
      float(ETinst[ETinst > -9999].min()),
      float(ETinst[ETinst > -9999].max()))

### 11.10 Save final instantaneous ET raster

In [ ]:
profile.update(dtype="float32", nodata=-9999.0)

ETinst_out = np.where(ETinst > -9999, ETinst, -9999).astype("float32")
ETinst_path = f"{out_dir}/ETinst_{date}_{hour_str}Z_{region}.tif"

write_raster(raster_path=ETinst_path, profile=profile, value=ETinst_out)

print("Saved:", ETinst_path)

### 11.11 DAILY ET SCALING (SEBAL – FIRST PASS)
------------------------------------------------------------

Daily ET is estimated from final instantaneous ET by assuming evaporative fraction remains approximately stable during the daytime period. For this winter Mississippi Delta scene, a 9-hour effective daylight scaling is applied. Because this is still a first-pass SEBAL solution, daily ET is constrained to a practical physical range (0–8 mm/day) to limit residual numerical spikes before aerodynamic stability refinement.

------------------------------------------------------------

In [ ]:
lambda_v = 2.45e6  # J/kg

# Rs_daily must be daily solar radiation (MJ/m²/day or W/m² integrated)
# If using daily Rn, replace below directly.

ET24 = np.full(ETinst.shape, -9999.0, dtype="float32")

valid_ET24 = (ETinst > -9999)

# Approximate daytime scaling factor for winter conditions
daylight_hours = 9.0

ET24[valid_ET24] = ETinst[valid_ET24] * daylight_hours

# Clamp to practical winter daily ET range
ET24[valid_ET24] = np.clip(ET24[valid_ET24], 0.0, 8.0)

print("Final ET24 min/max:",
      float(ET24[ET24 > -9999].min()),
      float(ET24[ET24 > -9999].max()))

### 11.12 Save final daily ET raster

In [ ]:
# 11.12 Save final daily ET raster

profile.update(dtype="float32", nodata=-9999.0)

ET24_out = np.where(ET24 > -9999, ET24, -9999).astype("float32")
ET24_path = f"{out_dir}/ET24_{date}_{hour_str}Z_{region}.tif"

write_raster(raster_path=ET24_path, profile=profile, value=ET24_out)

print("Saved:", ET24_path)